# Beginner E1: classify accents with i-vectors

This E1 extension uses the same `Singlish/data/audio` speaker folders as `E1_01_svm_beginner.ipynb`. It builds a **small, genuine i-vector pipeline** for learning, then trains a logistic-regression accent classifier.

The workflow is: audio → MFCC frames → universal background model (UBM) → clip statistics → total-variability matrix → i-vector → accent classifier. The UBM and total-variability model learn **without accent labels**; labels enter only at the classifier.

This notebook trains its own small extractor so every step is visible. It is a teaching experiment, not a production-quality i-vector system. A mature extractor normally uses far more speech. A high score here may reflect the difference between the MNSC and Common Voice corpora rather than accent alone.


In [ ]:
# Run in Google Colab. Restart the runtime if Colab asks after installation.
%pip -q install "librosa>=0.10.2" "scikit-learn>=1.3" matplotlib joblib soundfile


## 1. Point to the same audio as E1

Upload your `Singlish/data/audio` folder to Google Drive. It should contain `singapore/<speaker>/<clip>.wav` and `non_singapore/<speaker>/<clip>.wav`. Set `USE_GOOGLE_DRIVE = False` and change `PROJECT_ROOT` when running locally. Begin with 100 clips per class; set `MAX_CLIPS_PER_CLASS = None` to use all available clips.

Speaker folder names matter: they keep clips from the same voice on one side of the train/test split.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
DATA_ROOT = PROJECT_ROOT / 'data' / 'audio'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'ivector'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
MAX_CLIPS_PER_CLASS = 100  # None uses all clips.
SAMPLE_RATE = 16_000
MAX_SECONDS = 6
print('Audio:', DATA_ROOT)
print('Output:', OUTPUT_DIR)


In [ ]:
import json
from collections import Counter

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(RANDOM_SEED)


## 2. Find clips and keep their speaker IDs

Label `1` means the Singapore folder; label `0` means the non-Singapore folder. We sample speakers round-robin so the beginner subset does not consist of many clips from only a few voices. The group ID includes the class folder to avoid accidental collisions between anonymized speaker names.


In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}

def find_recordings(data_root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class directory: {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if path.suffix.lower() not in AUDIO_EXTENSIONS or not path.is_file():
                continue
            parts = path.relative_to(class_dir).parts
            if len(parts) < 2:
                raise ValueError(f'Clip needs a speaker folder: {path}')
            rows.append((path, label, f'{class_name}::{parts[0]}'))
    return rows

def select_recordings(rows, limit, seed):
    if limit is None:
        return rows
    if not isinstance(limit, int) or limit < 1:
        raise ValueError('MAX_CLIPS_PER_CLASS must be a positive integer or None')
    selection_rng = np.random.default_rng(seed)
    chosen = []
    for label in CLASS_LABELS.values():
        by_speaker = {}
        for row in rows:
            if row[1] == label:
                by_speaker.setdefault(row[2], []).append(row)
        speaker_ids = sorted(by_speaker)
        selection_rng.shuffle(speaker_ids)
        for speaker in speaker_ids:
            selection_rng.shuffle(by_speaker[speaker])
        while sum(row[1] == label for row in chosen) < limit:
            added = False
            for speaker in speaker_ids:
                if by_speaker[speaker]:
                    chosen.append(by_speaker[speaker].pop())
                    added = True
                    if sum(row[1] == label for row in chosen) == limit:
                        break
            if not added:
                break
    return sorted(chosen, key=lambda row: str(row[0]))

recordings = select_recordings(find_recordings(DATA_ROOT), MAX_CLIPS_PER_CLASS, RANDOM_SEED)
if not recordings:
    raise RuntimeError(f'No audio found below {DATA_ROOT}')
labels = np.array([row[1] for row in recordings], dtype=int)
groups = np.array([row[2] for row in recordings])
print('Clips:', Counter(labels))
print('Speakers:', {label: len(set(groups[labels == label])) for label in (0, 1)})


## 3. Split speakers **before** fitting the extractor

This order is essential. The UBM, total-variability matrix, frame scaler and final classifier all learn from training speakers only. The test clips are used only for final evaluation.


In [ ]:
speaker_counts = {label: len(set(groups[labels == label])) for label in (0, 1)}
if min(speaker_counts.values()) < 2:
    raise ValueError('Each class needs clips from at least two speakers')
n_splits = min(5, min(speaker_counts.values()))
splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(np.zeros(len(labels)), labels, groups))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert set(labels[train_idx]) == set(labels[test_idx]) == {0, 1}
print('Training clips:', len(train_idx), Counter(labels[train_idx]))
print('Test clips:', len(test_idx), Counter(labels[test_idx]))
print('Overlapping speakers:', len(set(groups[train_idx]) & set(groups[test_idx])))


## 4. Turn audio into short-frame MFCCs

Unlike the E1 baseline, we **keep each frame** rather than immediately taking clip-wide means. An i-vector extractor needs the distribution of frames in each recording. We trim leading and trailing quiet audio, cap duration at six seconds, compute 13 MFCCs and their first differences, then remove low-energy frames. Each clip becomes a `frames × 26` matrix. The same preprocessing applies to both classes.


In [ ]:
N_MFCC = 13
N_FFT = 400
HOP_LENGTH = 160

def audio_to_frames(path):
    waveform, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    waveform, _ = librosa.effects.trim(waveform, top_db=30)
    waveform = waveform[:SAMPLE_RATE * MAX_SECONDS]
    if len(waveform) < SAMPLE_RATE:
        raise ValueError('Less than one second of speech')
    peak = np.max(np.abs(waveform))
    if peak > 0:
        waveform = waveform / peak
    mfcc = librosa.feature.mfcc(y=waveform, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                n_fft=N_FFT, hop_length=HOP_LENGTH)
    delta = librosa.feature.delta(mfcc)
    rms = librosa.feature.rms(y=waveform, frame_length=N_FFT, hop_length=HOP_LENGTH).ravel()
    keep = rms[:mfcc.shape[1]] > max(float(np.max(rms)) * 0.03, 1e-5)
    frames = np.vstack((mfcc, delta)).T[keep].astype(np.float64)
    if len(frames) < 20 or not np.isfinite(frames).all():
        raise ValueError('Too few usable frames')
    return frames

frame_sets, kept_rows, kept_original_indices = [], [], []
for i, row in enumerate(recordings):
    try:
        frame_sets.append(audio_to_frames(row[0]))
        kept_rows.append(row)
        kept_original_indices.append(i)
    except Exception as exc:
        print('Skipping', row[0].name, ':', exc)
    if (i + 1) % 25 == 0 or i + 1 == len(recordings):
        print(f'Processed {i + 1}/{len(recordings)}')

original_to_kept = {old: new for new, old in enumerate(kept_original_indices)}
train_idx = np.array([original_to_kept[i] for i in train_idx if i in original_to_kept])
test_idx = np.array([original_to_kept[i] for i in test_idx if i in original_to_kept])
labels = np.array([row[1] for row in kept_rows], dtype=int)
groups = np.array([row[2] for row in kept_rows])
assert set(labels[train_idx]) == set(labels[test_idx]) == {0, 1}
assert not (set(groups[train_idx]) & set(groups[test_idx]))
print('One clip:', frame_sets[0].shape, '(frames × MFCC features)')


### Visual 1: the frame features entering the UBM

Each column is a short time frame from one **training** clip. Rows 0–12 are MFCCs and rows 13–25 are their first differences (deltas). Color shows each row standardized **for display only**. This reveals patterns that large-valued MFCC rows would otherwise hide. The i-vector model still uses the original feature values. The chart shows that the system receives a **sequence of frames**, rather than E1's one summary vector per clip.


In [ ]:
example_i = int(train_idx[0])
example_frames = frame_sets[example_i]
display_frames = (example_frames - example_frames.mean(axis=0)) / (example_frames.std(axis=0) + 1e-6)
fig, ax = plt.subplots(figsize=(10, 4))
image = ax.imshow(display_frames.T, origin='lower', aspect='auto', cmap='coolwarm',
                  vmin=-3, vmax=3,
                  extent=[0, len(example_frames) * HOP_LENGTH / SAMPLE_RATE, -0.5, 25.5])
ax.axhline(12.5, color='black', linewidth=1)
ax.set(xlabel='Approximate speech time (s)', ylabel='Feature row',
       title=f'MFCC and delta frames from a training clip ({kept_rows[example_i][0].name})')
ax.text(0.99, 0.23, 'MFCC 0–12', ha='right', transform=ax.transAxes)
ax.text(0.99, 0.75, 'Delta 13–25', ha='right', transform=ax.transAxes)
fig.colorbar(image, ax=ax, label='Standard deviations within row')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ivector_mfcc_frames.png', dpi=150)
plt.show()


## 5. Fit a universal background model (UBM)

The UBM is a Gaussian mixture fitted to **pooled training frames**, ignoring accent labels. Its components describe common kinds of acoustic frames. For each clip, we ask how strongly each frame belongs to each component. We fit a frame scaler on training frames as well.

`N_COMPONENTS=8` and the frame cap keep the demonstration manageable. Full i-vector systems usually use much larger background data and more mixture components.


In [ ]:
N_COMPONENTS = 8
MAX_UBM_FRAMES = 30_000

training_frames = np.concatenate([frame_sets[i] for i in train_idx], axis=0)
chosen = rng.choice(len(training_frames), size=min(MAX_UBM_FRAMES, len(training_frames)), replace=False)
ubm_frames = training_frames[chosen]
frame_scaler = StandardScaler().fit(ubm_frames)
ubm = GaussianMixture(n_components=N_COMPONENTS, covariance_type='diag',
                      reg_covar=1e-3, max_iter=100, random_state=RANDOM_SEED)
ubm.fit(frame_scaler.transform(ubm_frames))
print('UBM fitted on', len(ubm_frames), 'training frames')
print('Converged:', ubm.converged_)


## 6. Collect one set of UBM statistics per clip

For each mixture component, `N` is its expected frame count. `F` sums its assigned, **mean-centered** frames. These are the sufficient statistics used by the i-vector model. The UBM itself is fixed here; test clips only pass through it.


In [ ]:
def ubm_statistics(frames):
    scaled = frame_scaler.transform(frames)
    responsibilities = ubm.predict_proba(scaled)  # frames × components
    n = responsibilities.sum(axis=0)
    f = responsibilities.T @ scaled - n[:, None] * ubm.means_
    return n, f

stats = [ubm_statistics(frames) for frames in frame_sets]
print('N shape:', stats[0][0].shape, '| F shape:', stats[0][1].shape)


### Visual 2: UBM component occupancy

`N[c]` is the expected number of frames assigned to Gaussian component `c`. Dividing by the total gives the fraction of a clip's frames explained by that component. The chart compares one training clip from each folder to show what the UBM statistics mean. It is **not** evidence that a component represents an accent: it may capture phonetic content, speaker or recording conditions.


In [ ]:
component_ids = np.arange(N_COMPONENTS)
fig, ax = plt.subplots(figsize=(8, 4))
for label, offset, color, name in [(0, -0.18, 'tab:blue', 'Non-SG'),
                                   (1,  0.18, 'tab:orange', 'Singapore')]:
    example = next(int(i) for i in train_idx if labels[i] == label)
    counts = stats[example][0]
    ax.bar(component_ids + offset, counts / counts.sum(), width=0.36,
           color=color, label=f'{name} training clip')
ax.set(xticks=component_ids, xlabel='UBM Gaussian component',
       ylabel='Fraction of assigned frames', title='Example UBM component occupancy')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ivector_ubm_occupancy.png', dpi=150)
plt.show()


## 7. Learn a total-variability matrix, then extract i-vectors

The total-variability matrix `T` maps a low-dimensional hidden vector `w` to shifts in the UBM component means. We alternate two steps on **training clips only**:

1. **E-step:** estimate each clip's posterior mean and covariance for `w` from its `N` and `F` statistics;
2. **M-step:** update `T` from those posteriors.

The posterior mean is the clip's i-vector. It summarizes speaker, accent, channel and other variability together; it does **not** isolate accent by itself. The code is intentionally explicit so you can connect it to the diagram above.


In [ ]:
IVECTOR_DIM = 16
TV_ITERATIONS = 5
TV_REGULARIZATION = 1e-3
C, D = ubm.means_.shape
variances = ubm.covariances_  # diagonal Gaussian variances, C × D
T = rng.normal(0, 0.05, size=(C, D, IVECTOR_DIM))
identity = np.eye(IVECTOR_DIM)

def posterior(n, f, T_matrix):
    # L = I + sum_c N_c T_c^T Sigma_c^-1 T_c
    weighted_T = T_matrix / variances[:, :, None]
    precision = identity + np.einsum('c,cdr,cds->rs', n, T_matrix, weighted_T)
    covariance = np.linalg.inv(precision)
    # b = sum_c T_c^T Sigma_c^-1 F_c
    linear_term = np.einsum('cdr,cd->r', weighted_T, f)
    mean = covariance @ linear_term
    second_moment = covariance + np.outer(mean, mean)
    return mean, second_moment

for iteration in range(TV_ITERATIONS):
    numerator = np.zeros_like(T)                 # sum F_c E[w]^T
    denominator = np.zeros((C, IVECTOR_DIM, IVECTOR_DIM))  # sum N_c E[ww^T]
    for i in train_idx:
        n, f = stats[i]
        mean, second = posterior(n, f, T)
        numerator += f[:, :, None] * mean[None, None, :]
        denominator += n[:, None, None] * second[None, :, :]
    for component in range(C):
        T[component] = np.linalg.solve(
            denominator[component] + TV_REGULARIZATION * identity,
            numerator[component].T,
        ).T
    print(f'Total-variability iteration {iteration + 1}/{TV_ITERATIONS}')

def extract_ivector(n, f):
    vector, _ = posterior(n, f, T)
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 1e-12 else vector

X = np.vstack([extract_ivector(n, f) for n, f in stats])
assert X.shape == (len(frame_sets), IVECTOR_DIM) and np.isfinite(X).all()
print('i-vector matrix:', X.shape, '(clips × dimensions)')


## 8. Train a small accent classifier

The extractor used no accent labels. Logistic regression now learns a boundary between the two **labeled training groups**. `StandardScaler` and the classifier are fitted on training i-vectors only. This matches the small classifier-head idea in the Whisper notebook.


In [ ]:
classifier = Pipeline([
    ('scale', StandardScaler()),
    ('head', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000,
                                random_state=RANDOM_SEED)),
])
classifier.fit(X[train_idx], labels[train_idx])
print('Classifier trained on', len(train_idx), 'clips from', len(set(groups[train_idx])), 'speakers')


## 9. Evaluate speakers the model never trained on

Balanced accuracy weights both classes equally. Macro F1 averages their F1 scores. In the confusion matrix, **rows are true labels** and **columns are predictions**, ordered `[non-Singapore, Singapore]`. Use the same clips and split when comparing this notebook with E1 or Whisper; a shared seed alone does not guarantee an identical split.


In [ ]:
predictions = classifier.predict(X[test_idx])
probabilities = classifier.predict_proba(X[test_idx])[:, 1]
matrix = confusion_matrix(labels[test_idx], predictions, labels=[0, 1])
metrics = {
    'balanced_accuracy': float(balanced_accuracy_score(labels[test_idx], predictions)),
    'macro_f1': float(f1_score(labels[test_idx], predictions, average='macro')),
    'confusion_matrix': matrix.tolist(),
}
print(json.dumps(metrics, indent=2))
print(classification_report(labels[test_idx], predictions, labels=[0, 1],
                            target_names=['Non-Singapore', 'Singapore'], zero_division=0))
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap='Blues')
for (row, col), count in np.ndenumerate(matrix):
    ax.text(col, row, str(count), ha='center', va='center')
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Non-SG', 'Singapore'], yticklabels=['Non-SG', 'Singapore'],
       xlabel='Predicted', ylabel='True', title='Held-out speakers')
fig.colorbar(image, ax=ax)
plt.show()


## 10. Save the exact experiment

Save the fitted scaler, UBM, total-variability matrix, classifier and split together. `joblib` files should only be loaded from a source you trust. The `.npz` cache makes it easy to inspect or compare the i-vectors without running extraction again.


In [ ]:
artifact_path = OUTPUT_DIR / 'beginner_ivector_model.joblib'
cache_path = OUTPUT_DIR / 'beginner_ivectors.npz'
report_path = OUTPUT_DIR / 'beginner_ivector_report.json'
joblib.dump({'frame_scaler': frame_scaler, 'ubm': ubm, 'T': T,
             'classifier': classifier, 'sample_rate': SAMPLE_RATE,
             'ivector_dim': IVECTOR_DIM}, artifact_path)
np.savez_compressed(cache_path, X=X, labels=labels, groups=groups,
                    paths=np.array([str(row[0]) for row in kept_rows]),
                    train_idx=train_idx, test_idx=test_idx)
report = {'settings': {'seed': RANDOM_SEED, 'max_clips_per_class': MAX_CLIPS_PER_CLASS,
                       'ubm_components': N_COMPONENTS, 'ivector_dim': IVECTOR_DIM,
                       'tv_iterations': TV_ITERATIONS},
          'split': {'train_clips': len(train_idx), 'test_clips': len(test_idx),
                    'train_speakers': len(set(groups[train_idx])),
                    'test_speakers': len(set(groups[test_idx])),
                    'speaker_overlap': len(set(groups[train_idx]) & set(groups[test_idx]))},
          'metrics': metrics}
report_path.write_text(json.dumps(report, indent=2) + '\n')
print('Saved:', artifact_path, cache_path, report_path, sep='\n')


## What the result means

If the i-vector classifier performs well on held-out speakers, its representation contains information that separates these two datasets. It does **not yet prove** that it recognizes Singapore English accent: MNSC and Common Voice also differ in microphones, prompts and recording style. A stronger follow-up evaluates both accents under matched recording conditions, adds a validation split for tuning, and compares all models on the **same saved split**.

Try changing `N_COMPONENTS`, `IVECTOR_DIM` or `MAX_CLIPS_PER_CLASS` one at a time. The test set should remain untouched while selecting settings; use a speaker-disjoint validation set for a formal model comparison.
